<a href="https://colab.research.google.com/github/gabrielaaguiv5/ProyectoFinal2/blob/main/notebooks/AnalisisExportacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# SmartChat Insight
# Análisis de clientes recurrentes, perdidos y productos más solicitados
# =========================================================

# Importar librerías
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# Definir rutas
clean_path = Path("../data/cleaned/whatsapp_clean.csv")
output_path = Path("../data/outputs/")
output_path.mkdir(parents=True, exist_ok=True)

# Cargar datos limpios
df = pd.read_csv(clean_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Datos cargados correctamente")
print(f"Total de mensajes: {len(df)}")
df.head(5)


FileNotFoundError: [Errno 2] No such file or directory: '../data/cleaned/whatsapp_clean.csv'

In [ ]:
# Identificar participantes únicos
clientes = df["user"].unique()
print(f"Total de usuarios detectados: {len(clientes)}")

# Calcular número de mensajes, primera y última fecha
activity = (
    df.groupby("user")
      .agg(
          mensajes=("message", "count"),
          primer_contacto=("date", "min"),
          ultimo_contacto=("date", "max")
      )
      .reset_index()
)

# Calcular días desde el último mensaje
activity["dias_desde_ultimo"] = (df["date"].max() - activity["ultimo_contacto"]).dt.days

# Clasificar clientes
def clasificar_cliente(dias):
    if dias <= 15:
        return "Frecuente"
    elif dias <= 45:
        return "Inactivo reciente"
    else:
        return "Perdido"

activity["estado"] = activity["dias_desde_ultimo"].apply(clasificar_cliente)

activity.head(10)


In [ ]:
# Palabras clave de productos (se adaptan según cada negocio)
productos = [
    "vidrio", "fachada", "aluminio", "puerta", "ventana",
    "baño", "división", "pasamanos", "pérgola", "instalador"
]

# Contar menciones de cada producto
conteo_productos = {}
for p in productos:
    conteo_productos[p] = df["message"].str.contains(p, case=False, na=False).sum()

df_productos = pd.DataFrame(list(conteo_productos.items()), columns=["producto", "menciones"])
df_productos = df_productos.sort_values(by="menciones", ascending=False)
df_productos


In [ ]:
# Gráfico de clientes según estado
plt.figure(figsize=(6,4))
activity["estado"].value_counts().plot(kind="bar")
plt.title("Distribución de Clientes por Estado")
plt.xlabel("Estado")
plt.ylabel("Número de clientes")
plt.show()

# Gráfico de productos más mencionados
plt.figure(figsize=(8,4))
df_productos.plot(kind="barh", x="producto", y="menciones", legend=False)
plt.title("Productos más mencionados en los chats")
plt.xlabel("Número de menciones")
plt.ylabel("Producto")
plt.show()


In [ ]:
activity.to_csv(output_path / "clientes.csv", index=False, encoding="utf-8-sig")
df_productos.to_csv(output_path / "productos.csv", index=False, encoding="utf-8-sig")

print("✅ Archivos exportados para Power BI:")
print(f"- {output_path / 'clientes.csv'}")
print(f"- {output_path / 'productos.csv'}")

In [ ]:
# =========================================================
# SmartChat Insight
# Exportación y preparación de datasets finales para Power BI
# =========================================================

# Importar librerías
import pandas as pd
from pathlib import Path

# Rutas de entrada y salida
input_path = Path("../data/outputs/")
final_path = Path("../data/outputs/final/")
final_path.mkdir(parents=True, exist_ok=True)

# Cargar archivos de análisis previos
clientes = pd.read_csv(input_path / "clientes.csv")
productos = pd.read_csv(input_path / "productos.csv")

print(f"Clientes cargados: {len(clientes)} registros")
print(f"Productos cargados: {len(productos)} registros")

clientes.head(5)

In [ ]:
# Normalizar nombres de columnas a formato uniforme para Power BI
clientes.columns = clientes.columns.str.lower().str.replace(" ", "_")
productos.columns = productos.columns.str.lower().str.replace(" ", "_")

# Asegurar tipos de datos correctos
clientes["primer_contacto"] = pd.to_datetime(clientes["primer_contacto"], errors="coerce")
clientes["ultimo_contacto"] = pd.to_datetime(clientes["ultimo_contacto"], errors="coerce")
clientes["dias_desde_ultimo"] = clientes["dias_desde_ultimo"].astype(int)

In [ ]:
# Crear un resumen con totales por estado
resumen_clientes = (
    clientes.groupby("estado")
    .agg(
        total_clientes=("user", "count"),
        promedio_mensajes=("mensajes", "mean"),
        dias_promedio_inactividad=("dias_desde_ultimo", "mean")
    )
    .reset_index()
)

# Agregar fecha de generación del reporte
from datetime import datetime
resumen_clientes["fecha_reporte"] = datetime.now().strftime("%Y-%m-%d")
resumen_clientes


In [ ]:
clientes.to_csv(final_path / "clientes_powerbi.csv", index=False, encoding="utf-8-sig", sep=";")
productos.to_csv(final_path / "productos_powerbi.csv", index=False, encoding="utf-8-sig", sep=";")
resumen_clientes.to_csv(final_path / "resumen_clientes.csv", index=False, encoding="utf-8-sig", sep=";")

print("Archivos finales generados:")
for file in final_path.glob("*.csv"):
    print(f"- {file}")